# Populate Tier-1 feature cache (ESM-2, AlphaFold-DB, MACE-OFF)

**Run on Colab** with a GPU (A100 / L4 / T4 / Blackwell Pro all work).
Populates the three Tier-1 parquet files under
`cell_sim/features/cache/`:

| Extractor        | Status on a JCVI-Syn3A run                          |
|------------------|-----------------------------------------------------|
| ESM-2 (650M)     | full per-protein 1280-dim embeddings, ~18 s on A100 |
| AlphaFold-DB     | empty — JCVI-Syn3A (taxid 2144189) is a synthetic organism and UniProt does not index its proteome, so AFDB has nothing to fetch. Session 15 will wire in a curated mapping from the Luthey-Schulten supplement. |
| MACE-OFF (small) | empty — the Syn3A SBML encodes metabolites as species IDs (`M_atp_c`), not SMILES. Session 15 will wire in a curated SMILES map (KEGG / ChEBI) before invoking the backend. |

So: **ESM-2 is the real deliverable this session.** The other two
parquets are registered + tracked by the manifest but carry no
information content yet. That keeps the three-source manifest
consistent for Session 15 without manufacturing fake data.

Non-goals (deliberately): no detector changes, no MCC measurement,
no model weights committed to the repo.


In [ ]:
# Cell 2 — install the GPU-side deps. These are intentionally NOT in
# cell_sim/requirements.txt — see the comment there.
#
# The install is split into two groups:
#   1. Core stack (ESM-2 + AlphaFold + framework). Required.
#   2. MACE-OFF stack. Optional — mace-torch pins an older e3nn, so
#      we let pip resolve mace's own compatible version and tolerate
#      a failure (ESM-2 + AlphaFold still populate).

# --- Group 1: core stack (ESM-2, AlphaFold, supporting libs) ---
!pip install -q \
    "torch>=2.2" \
    "transformers>=4.40" \
    "biopython>=1.83" \
    "rdkit>=2023.9" \
    "ase>=3.22" \
    "pyarrow>=14" \
    "pandas>=2" \
    "numpy>=1.24"

# --- Group 2: MACE-OFF (optional, may fail; ESM-2 / AlphaFold survive) ---
# NOTE: do NOT pin e3nn here — mace-torch's own metadata selects a
# compatible version. Pinning e3nn>=0.5.1 breaks resolution on
# every mace-torch 0.3.x release (they pin e3nn<0.5).
import subprocess, sys
_mace_status = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mace-torch"],
    capture_output=True, text=True,
)
MACE_AVAILABLE = (_mace_status.returncode == 0)
if not MACE_AVAILABLE:
    print("[warn] mace-torch install failed; MACE-OFF cell will be skipped.")
    print("       last pip stderr line:")
    for line in _mace_status.stderr.strip().splitlines()[-3:]:
        print(f"         {line}")
else:
    print("mace-torch install OK")

import torch
print(f"\ntorch {torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  VRAM: {props.total_memory / 1024**3:.1f} GiB")
else:
    print("  CPU only — ESM-2 will run but will take much longer.")
print(f"MACE_AVAILABLE = {MACE_AVAILABLE}")


In [ ]:
# Cell 3 — clone (or refresh) the project repo at the current branch HEAD.
#
# Re-run behavior: if /content/cell already exists from a previous
# kernel, we FETCH + RESET --hard so the working tree matches the
# live branch tip. Without this, Colab users hit a silent bug where
# a stale clone from an earlier session misses recent commits
# (e.g. the curated SMILES map, the compute_tm monkey-patch) and the
# extractor cells fail or write stale data. Seen in Session-15's
# 3-run debugging loop.
import os, subprocess

BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"


def _run(cmd, cwd=None, check=True):
    res = subprocess.run(
        cmd, cwd=cwd, capture_output=True, text=True,
    )
    if res.stdout.strip():
        print(res.stdout.rstrip())
    if res.stderr.strip():
        print(res.stderr.rstrip())
    if check and res.returncode != 0:
        raise RuntimeError(f"{cmd!r} exit {res.returncode}")
    return res


if not os.path.isdir(REPO_DIR):
    print(f"cloning {REPO_URL} (branch {BRANCH}) into {REPO_DIR} ...")
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    print(f"{REPO_DIR} exists; refreshing to origin/{BRANCH} ...")
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)

%cd /content/cell

import sys
sys.path.insert(0, "/content/cell")
print("CWD:", os.getcwd())
_run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)


In [ ]:
# Cell 4 — stage the Luthey-Schulten input data, then load Syn3A CDS
# sequences from the staged GenBank file.
#
# The 5 upstream files (syn3A.gb, kinetic_params.xlsx,
# initial_concentrations.xlsx, complex_formation.xlsx, Syn3A_updated.xml)
# are NOT committed to Nikku03/cell — they live under
# cell_sim/data/Minimal_Cell_ComplexFormation/input_data/ which is
# gitignored (see memory_bank/data/STAGING.md). We fetch them from
# the upstream repo here, idempotent (skipped if already present).
import urllib.request
from pathlib import Path

STAGED_DIR = Path(
    "cell_sim/data/Minimal_Cell_ComplexFormation/input_data"
)
STAGED_DIR.mkdir(parents=True, exist_ok=True)
UPSTREAM = (
    "https://raw.githubusercontent.com/Luthey-Schulten-Lab/"
    "Minimal_Cell_ComplexFormation/master/input_data"
)
STAGED_FILES = [
    "syn3A.gb",
    "kinetic_params.xlsx",
    "initial_concentrations.xlsx",
    "complex_formation.xlsx",
    "Syn3A_updated.xml",
]
for fname in STAGED_FILES:
    dest = STAGED_DIR / fname
    if dest.exists() and dest.stat().st_size > 0:
        continue
    url = f"{UPSTREAM}/{fname}"
    print(f"staging {fname} from upstream...")
    urllib.request.urlretrieve(url, dest)
print(f"staged {len(STAGED_FILES)} files under {STAGED_DIR}")

# --- load Syn3A CDS translations ---
from Bio import SeqIO
import pandas as pd

GB_PATH = STAGED_DIR / "syn3A.gb"
records = list(SeqIO.parse(GB_PATH, "genbank"))
assert len(records) == 1, f"expected 1 record, got {len(records)}"
rec = records[0]
print(f"accession={rec.id}  length={len(rec.seq):,} bp")

cds_rows = []
for feat in rec.features:
    if feat.type != "CDS":
        continue
    locus = (feat.qualifiers.get("locus_tag") or [""])[0]
    gene_name = (feat.qualifiers.get("gene") or [""])[0]
    translation = (feat.qualifiers.get("translation") or [""])[0]
    protein_id = (feat.qualifiers.get("protein_id") or [""])[0]
    product = (feat.qualifiers.get("product") or [""])[0]
    if not locus or not translation:
        continue
    cds_rows.append({
        "locus_tag": locus,
        "gene_name": gene_name,
        "protein_id": protein_id,
        "product": product,
        "sequence": translation.upper(),
    })
cds_df = pd.DataFrame(cds_rows)
print(f"\n{len(cds_df)} CDS with translations")
print(cds_df.head(3))
dnaA_match = cds_df.loc[cds_df["gene_name"] == "dnaA"]
if len(dnaA_match):
    dnaA_seq = dnaA_match["sequence"].iloc[0]
    print(f"\ndnaA length: {len(dnaA_seq)} aa; starts: {dnaA_seq[:10]}")


In [ ]:
# Cell 5 — ESM-2 650M embeddings for every CDS with a translation.
import hashlib, time
from pathlib import Path
from cell_sim.features.extractors import ESM2Extractor
from cell_sim.features.batched_inference import (
    BatchedInferenceConfig,
)

CACHE_DIR = Path("cell_sim/features/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

esm = ESM2Extractor()
cfg = BatchedInferenceConfig(
    batch_size=16,
    device="auto",
    dtype="float16" if torch.cuda.is_available() else "float32",
    max_seq_length=1022,
    progress=True,
)

t0 = time.perf_counter()
esm_feats = esm.extract(cds_df[["locus_tag", "sequence"]], cfg)
esm_wall = time.perf_counter() - t0
esm_path, esm_sha = esm.write_cache(esm_feats, cache_dir=CACHE_DIR)
print(f"ESM-2 done in {esm_wall:.1f}s  parquet={esm_path}")
print(f"  rows={len(esm_feats)}  dims={len(esm.feature_cols)}")
print(f"  sha256={esm_sha}")
print("\nfirst 5 locus tags' embedding L2 norms:")
for locus in esm_feats.index[:5]:
    vec = esm_feats.loc[locus]
    print(f"  {locus}  ||x||={(vec ** 2).sum() ** 0.5:.3f}")


In [ ]:
# Cell 6 — ESMFold v1 per-protein structural descriptors.
#
# Session-15 pivot from AlphaFold-DB: JCVI-Syn3A (taxid 2144189) is
# not indexed in UniProt, so AFDB has no structures keyed to Syn3A
# accessions. ESMFold takes the AA sequence directly.
#
# PRECISION: we run the WHOLE model in fp32 (LM + trunk). The
# official HuggingFace mixed-precision recipe (LM fp16 + trunk fp32)
# produced has_structure=1 but all-NaN pLDDT in Session 16 run #3 —
# the per-residue confidence head is more fp16-sensitive than the
# coordinate head, so the PDB parsed but the B-factor column was NaN.
# Running fp32 end-to-end needs ~24 GiB VRAM (fits on A100 / L4 /
# Blackwell Pro) and is only ~1.5x slower than mixed precision.
#
# CHECKPOINTING: we iterate gene-by-gene and pickle a running DF
# every CHECKPOINT_EVERY genes to /content/esmfold_checkpoint.pkl.
# A kernel restart or OOM only loses at most CHECKPOINT_EVERY genes;
# re-running the cell resumes from the checkpoint.
#
# PRE-FLIGHT: we fold ONE gene and assert that plddt_mean is FINITE
# (not just that has_structure == 1.0) before committing to the full
# loop. The has_structure flag is set whenever a PDB parses, even if
# its B-factor column is entirely NaN — which is exactly what fp16
# produced last run, silently poisoning 450 rows.
import math
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd

from cell_sim.features.extractors import ESMFoldExtractor
from cell_sim.features.extractors.esmfold_extractor import (
    _no_structure_row,
    _features_from_pdb,
    _rename_af_to_esmfold,
)
from cell_sim.features.batched_inference import BatchedInferenceConfig

CACHE_DIR = Path("cell_sim/features/cache")
CHECKPOINT_PATH = Path("/content/esmfold_checkpoint.pkl")
CHECKPOINT_EVERY = 10
ESMFOLD_DTYPE = "float32"   # "float16" is unsafe — see note above.

ef_input = cds_df[["locus_tag", "sequence"]].copy()
ef_input = ef_input[ef_input["sequence"].str.strip().astype(bool)]
ef_input = ef_input.reset_index(drop=True)
print(f"ESMFold inputs: {len(ef_input)}/{len(cds_df)} CDS have a sequence")

ef = ESMFoldExtractor()
ef._ensure_loaded(BatchedInferenceConfig(
    batch_size=1, device="cuda", dtype=ESMFOLD_DTYPE, max_seq_length=1024,
))

# --- pre-flight on the first non-trivial sequence ------------------
print(f"\n[pre-flight] folding first gene in dtype={ESMFOLD_DTYPE}...")
first_row = ef_input.iloc[0]
first_seq = first_row["sequence"].strip().upper().rstrip("*")[:1024]
t_pre = time.perf_counter()
try:
    pdb_str = ef._infer_pdb(first_seq)
    af_row = _features_from_pdb(pdb_str.encode("utf-8"))
    sample_feat = _rename_af_to_esmfold(af_row)
except Exception as exc:  # noqa: BLE001
    print(f"[pre-flight] FAILED on {first_row['locus_tag']}: "
          f"{type(exc).__name__}: {exc}")
    raise

has_struct = sample_feat.get("esmfold_has_structure", 0.0)
plddt = sample_feat.get("esmfold_plddt_mean", float("nan"))
assert has_struct == 1.0, (
    f"pre-flight: has_structure={has_struct} (expected 1.0); "
    f"PDB parse failed"
)
assert isinstance(plddt, (int, float)) and math.isfinite(plddt), (
    f"pre-flight: plddt_mean={plddt} — model produced a structure "
    f"but the pLDDT head emitted NaN. This is the fp16 numerical-"
    f"instability bug from Session 16 run #3. Stop and investigate "
    f"before committing to a 30-60 min loop."
)
print(f"[pre-flight] OK  locus={first_row['locus_tag']}  "
      f"len={len(first_seq)}aa  plddt={plddt:.1f}  "
      f"took {time.perf_counter() - t_pre:.1f}s")

# --- checkpointed main loop ----------------------------------------
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open("rb") as fh:
        checkpoint = pickle.load(fh)
    processed_loci = set(checkpoint["rows"].keys())
    print(f"\n[resume] checkpoint found with {len(processed_loci)} "
          f"genes already processed; skipping those")
else:
    checkpoint = {"rows": {}, "started_at": time.time()}
    processed_loci = set()

# Seed the checkpoint with the pre-flight result so we don't re-fold
# the first gene.
if first_row["locus_tag"] not in processed_loci:
    checkpoint["rows"][first_row["locus_tag"]] = sample_feat
    processed_loci.add(first_row["locus_tag"])

t0 = time.perf_counter()
for i, row in ef_input.iterrows():
    locus = row["locus_tag"]
    if locus in processed_loci:
        continue

    seq = row["sequence"].strip().upper().rstrip("*")
    if not seq or seq == "NAN":
        checkpoint["rows"][locus] = _no_structure_row()
    else:
        if len(seq) > 1024:
            seq = seq[:1024]
        try:
            pdb_str = ef._infer_pdb(seq)
            af_row = _features_from_pdb(pdb_str.encode("utf-8"))
            checkpoint["rows"][locus] = _rename_af_to_esmfold(af_row)
        except Exception as exc:  # noqa: BLE001
            print(f"  [skip] {locus} len={len(seq)}: "
                  f"{type(exc).__name__}: {exc}")
            checkpoint["rows"][locus] = _no_structure_row()

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(ef_input):
        with CHECKPOINT_PATH.open("wb") as fh:
            pickle.dump(checkpoint, fh)
        elapsed = time.perf_counter() - t0
        n_done = len(checkpoint["rows"])
        rate = n_done / max(1e-6, elapsed)
        eta_s = (len(ef_input) - n_done) / max(1e-6, rate)
        print(f"  [ckpt] {n_done}/{len(ef_input)}  "
              f"elapsed={elapsed:.0f}s  eta={eta_s:.0f}s  "
              f"rate={rate:.2f} gene/s")

ef_wall = time.perf_counter() - t0

# --- assemble dataframe in the canonical row order & write -------
rows = [checkpoint["rows"][lt] for lt in ef_input["locus_tag"]]
ef_feats = pd.DataFrame(rows, columns=ef.feature_cols)
ef_feats.index = pd.Index(
    ef_input["locus_tag"].tolist(), name="locus_tag",
)

n_struct = int(ef_feats["esmfold_has_structure"].sum())
n_finite_plddt = int(np.isfinite(ef_feats["esmfold_plddt_mean"]).sum())
ef_path, ef_sha = ef.write_cache(ef_feats, cache_dir=CACHE_DIR)
print(f"\nESMFold done in {ef_wall:.1f}s  parquet={ef_path}")
print(f"  rows={len(ef_feats)}  structures predicted={n_struct}  "
      f"finite pLDDT={n_finite_plddt}")
print(f"  sha256={ef_sha}")
if n_finite_plddt < n_struct:
    print(f"  [WARN] {n_struct - n_finite_plddt} rows have "
          f"has_structure=1 but NaN pLDDT — fp16 leak?")

if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print(f"  removed checkpoint {CHECKPOINT_PATH}")


In [ ]:
# Cell 7 — MACE-OFF BDE-derived k_cat refinement.
#
# Reactions whose ALL substrates are in the curated SMILES map
# trigger real BDE -> kcat aggregates. Reactions with any unmapped
# substrate are skipped; MACE writes NaN rows for their enzymes and
# the parquet tracks them via mace_has_estimate == 0.0.
#
# Path resolution: the SMILES CSV lives under memory_bank/data/ in
# the repo root. We anchor to the repo root via the cell_sim module
# location, so a rogue %cd or kernel-restart-with-different-cwd
# can't silently break the map lookup (previously caused an empty
# placeholder parquet because the relative path missed).
import time
import numpy as np
import pandas as pd
from pathlib import Path

import cell_sim as _cell_sim_pkg
from cell_sim.features.extractors import MaceOffExtractor
from cell_sim.layer3_reactions.sbml_parser import parse_sbml

# cell_sim is a namespace package (no __init__.py), so __file__ is
# None and we anchor via __path__[0] instead. That's a list of every
# directory contributing to the namespace; the first entry is always
# the importable on-disk location for our layout.
REPO_ROOT = Path(list(_cell_sim_pkg.__path__)[0]).resolve().parent
SMILES_CSV = REPO_ROOT / "memory_bank/data/syn3a_species_smiles.csv"
SBML_PATH = (
    REPO_ROOT
    / "cell_sim/data/Minimal_Cell_ComplexFormation/input_data"
    / "Syn3A_updated.xml"
)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"SMILES_CSV  = {SMILES_CSV}  exists={SMILES_CSV.exists()}")
print(f"SBML_PATH   = {SBML_PATH}  exists={SBML_PATH.exists()}")
print(f"MACE_AVAILABLE = {MACE_AVAILABLE}")

mace = MaceOffExtractor(model="small", device="auto")
HAVE_CURATED_SMILES_MAP = SMILES_CSV.exists()

if HAVE_CURATED_SMILES_MAP:
    smiles_df = pd.read_csv(SMILES_CSV)
    SUBSTRATE_SMILES_MAP: dict[str, str] = dict(
        zip(smiles_df["species_id"], smiles_df["smiles"])
    )
    print(f"loaded {len(SUBSTRATE_SMILES_MAP)} species_id -> SMILES entries")
else:
    print(f"[warn] smiles map not found at {SMILES_CSV}")

t0 = time.perf_counter()
if not HAVE_CURATED_SMILES_MAP or not MACE_AVAILABLE:
    if not MACE_AVAILABLE:
        reason = "mace-torch install failed (see cell 2 pip output)"
    else:
        reason = f"smiles map missing ({SMILES_CSV})"
    print(f"skipping MACE-OFF bootstrap: {reason}")
    mace_feats = pd.DataFrame(
        {c: [np.nan] * len(cds_df) for c in mace.feature_cols},
    )
    mace_feats["mace_has_estimate"] = 0.0
    mace_feats.index = pd.Index(cds_df["locus_tag"].tolist(), name="locus_tag")
else:
    sbml = parse_sbml(SBML_PATH)

    mace_rows: list[dict] = []
    n_rxn_full = n_rxn_partial = n_rxn_none = 0
    for rxn in sbml.reactions.values():
        if not rxn.gene_associations:
            continue
        all_mapped = all(
            s in SUBSTRATE_SMILES_MAP for s in rxn.reactants
        )
        any_mapped = any(
            s in SUBSTRATE_SMILES_MAP for s in rxn.reactants
        )
        if all_mapped and rxn.reactants:
            n_rxn_full += 1
        elif any_mapped:
            n_rxn_partial += 1
        else:
            n_rxn_none += 1
            continue
        for gene_id in rxn.gene_associations:
            locus = gene_id.replace("G_MMSYN1_", "JCVISYN3A_")
            for sp_id in rxn.reactants:
                smi = SUBSTRATE_SMILES_MAP.get(sp_id)
                if not smi:
                    continue
                mace_rows.append({
                    "locus_tag": locus,
                    "enzyme_name": rxn.short_name,
                    "reaction_class": "metabolic",
                    "substrate_smiles": smi,
                })

    print(f"reaction coverage: {n_rxn_full} full / {n_rxn_partial} partial "
          f"/ {n_rxn_none} skipped (no substrate mapped)")
    print(f"  -> {len(mace_rows)} (locus, substrate) rows fed to MACE-OFF")

    mace_input = pd.DataFrame(mace_rows)
    if len(mace_input) == 0:
        print("[warn] no (locus, substrate) rows produced despite map + "
              "MACE install — check SBML parsing / species_id format.")
        mace_feats = pd.DataFrame(
            {c: [np.nan] * len(cds_df) for c in mace.feature_cols},
        )
        mace_feats["mace_has_estimate"] = 0.0
        mace_feats.index = pd.Index(cds_df["locus_tag"].tolist(),
                                     name="locus_tag")
    else:
        mace_feats = mace.extract(mace_input)
        # Fill unreferenced loci with NaN rows so the parquet shape
        # matches cds_df and the registry contract holds.
        missing = [lt for lt in cds_df["locus_tag"]
                   if lt not in mace_feats.index]
        if missing:
            pad = pd.DataFrame(
                {c: [np.nan] * len(missing) for c in mace.feature_cols},
            )
            pad["mace_has_estimate"] = 0.0
            pad.index = pd.Index(missing, name="locus_tag")
            mace_feats = pd.concat([mace_feats, pad])

mace_wall = time.perf_counter() - t0
mace_path, mace_sha = mace.write_cache(mace_feats, cache_dir=CACHE_DIR)
n_est = int(mace_feats["mace_has_estimate"].sum())
print(f"MACE-OFF done in {mace_wall:.1f}s  parquet={mace_path}")
print(f"  rows={len(mace_feats)}  enzymes_with_estimate={n_est}")
print(f"  sha256={mace_sha}")


In [ ]:
# Cell 8 — refresh manifest.json and verify every registered source.
# write_cache() already adds each parquet as it lands, but we re-load
# and re-verify here so the user sees the full manifest state in one
# place and we loudly flag any drift between manifest ↔ disk.
#
# Failure modes caught:
#   - manifest lists a source but the parquet is missing on disk
#     (extractor cell crashed mid-run or the file was deleted)
#   - parquet on disk has a different sha than the manifest claims
#     (stale manifest or manual edit)
#   - a parquet is on disk but no manifest entry exists
#     (add_source forgot to fire)
from cell_sim.features.cache_manifest import CachedFeatureManifest

MANIFEST_PATH = CACHE_DIR / "manifest.json"
manifest = CachedFeatureManifest.load(MANIFEST_PATH)

disk_parquets = {p.stem for p in CACHE_DIR.glob("*.parquet")}
manifest_sources = set(manifest.sources.keys())
all_names = sorted(disk_parquets | manifest_sources)

print(f"manifest at {MANIFEST_PATH}")
print(f"  tracked sources: {sorted(manifest_sources)}")
print(f"  disk parquets  : {sorted(disk_parquets)}")

n_ok = n_missing = n_stale = n_orphan = 0
for name in all_names:
    parquet = CACHE_DIR / f"{name}.parquet"
    in_manifest = name in manifest_sources
    on_disk = parquet.exists() and parquet.stat().st_size > 0
    if in_manifest and on_disk:
        ok = manifest.verify(name, parquet)
        entry = manifest.sources[name]
        status = "OK" if ok else "SHA-MISMATCH"
        if not ok:
            n_stale += 1
        else:
            n_ok += 1
        print(f"  [{status:12s}] {name:20s}  v{entry['version']}  "
              f"rows={entry['rows']:<6}  sha={entry['sha256'][:12]}...")
    elif in_manifest and not on_disk:
        n_missing += 1
        print(f"  [MISSING-DISK] {name:20s}  in manifest but parquet absent")
    else:  # on disk, not in manifest
        n_orphan += 1
        print(f"  [ORPHAN     ]  {name:20s}  parquet present, no manifest entry")

print()
print(f"summary: ok={n_ok}  stale={n_stale}  "
      f"missing-on-disk={n_missing}  orphan={n_orphan}")

if n_stale or n_missing:
    print()
    print("[WARN] manifest/disk drift detected. Re-run the affected "
          "extractor cell before shipping, or the downstream detector "
          "stack will load inconsistent data.")


In [ ]:
# Cell 9 — ship the populated cache back to repo/machine. Modes:
#   "download"    — browser-download every parquet + manifest (default)
#   "drive"       — copies to /content/drive/MyDrive/cell_sim_cache/
#   "github_pat"  — automated git add -f + commit + push
#
# CRITICAL: PARQUET_NAMES is auto-discovered from the UNION of
#   (a) every *.parquet file physically present in CACHE_DIR, and
#   (b) every source registered in manifest.json.
# This fixes a Session-15 bug where a hardcoded tuple silently dropped
# esmfold_v1.parquet from 3 consecutive pushes because the list was
# missing that name. Auto-discovery means new extractors never need
# this cell touched again.

import os, subprocess
from pathlib import Path
from cell_sim.features.cache_manifest import CachedFeatureManifest

OUTPUT_MODE = "download"   # options: "download" | "drive" | "github_pat"
COMMIT_MESSAGE = (
    "Populate Tier 1 cache (ESM-2, ESMFold-v1, MACE-OFF)"
)
BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"

MANIFEST_PATH = CACHE_DIR / "manifest.json"
manifest = CachedFeatureManifest.load(MANIFEST_PATH)

disk_parquets = {p.stem for p in CACHE_DIR.glob("*.parquet")}
manifest_parquets = set(manifest.sources.keys())
PARQUET_NAMES = tuple(sorted(disk_parquets | manifest_parquets))
PARQUET_PATHS = [CACHE_DIR / f"{name}.parquet" for name in PARQUET_NAMES]

print(f"discovered {len(PARQUET_NAMES)} parquet(s) to ship:")
for name in PARQUET_NAMES:
    path = CACHE_DIR / f"{name}.parquet"
    on_disk = path.exists() and path.stat().st_size > 0
    in_manifest = name in manifest_parquets
    size_mb = path.stat().st_size / 1024**2 if on_disk else 0.0
    flag = "OK " if (on_disk and in_manifest) else "WARN"
    print(f"  [{flag}] {name:20s}  disk={on_disk}  "
          f"manifest={in_manifest}  size={size_mb:.1f} MiB")

missing = [p for p in PARQUET_PATHS
           if not (p.exists() and p.stat().st_size > 0)]
if missing:
    print()
    print(f"[abort] {len(missing)} parquet(s) registered in manifest but "
          f"missing on disk:")
    for p in missing:
        print(f"  - {p}")
    print("Re-run the cell that produced the missing extractor, then "
          "re-run this cell.")
    raise SystemExit(1)


def _do_download() -> None:
    from google.colab import files  # type: ignore
    for p in PARQUET_PATHS:
        files.download(str(p))
    files.download(str(MANIFEST_PATH))
    print(f"{len(PARQUET_PATHS) + 1} downloads triggered "
          f"({len(PARQUET_PATHS)} parquets + manifest.json).")


def _do_drive() -> None:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:  # noqa: BLE001
        print(f"Drive mount failed: {type(exc).__name__}: {exc}")
        print("Flip OUTPUT_MODE to 'download' or 'github_pat' and rerun.")
        return
    import shutil
    dest = Path("/content/drive/MyDrive/cell_sim_cache")
    dest.mkdir(parents=True, exist_ok=True)
    for p in PARQUET_PATHS:
        shutil.copy(p, dest / p.name)
    shutil.copy(MANIFEST_PATH, dest / "manifest.json")
    print(f"Copied {len(PARQUET_PATHS)} parquets + manifest.json to {dest}")


def _do_github_pat() -> None:
    pat = os.environ.get("GITHUB_PAT", "").strip()
    if not pat:
        print("GITHUB_PAT is not set in this kernel. Either:")
        print("  1. Colab sidebar -> Secrets -> add 'GITHUB_PAT', then:")
        print("       from google.colab import userdata")
        print("       import os")
        print("       os.environ['GITHUB_PAT'] = userdata.get('GITHUB_PAT')")
        print("  2. Or inline: %env GITHUB_PAT=ghp_xxx")
        print("Then re-run this cell.")
        return

    def run(cmd, check=True):
        res = subprocess.run(cmd, capture_output=True, text=True)
        if res.stdout.strip():
            print(res.stdout.rstrip())
        if res.stderr.strip():
            print(res.stderr.rstrip())
        if check and res.returncode != 0:
            raise RuntimeError(f"{cmd!r} exit {res.returncode}")
        return res

    run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
    run(["git", "config", "user.name", "cell-sim-bot"])
    paths = [str(p) for p in PARQUET_PATHS] + [str(MANIFEST_PATH)]
    # -f because the parquets are gitignored by default.
    run(["git", "add", "-f", *paths])
    status = run(["git", "status", "--porcelain"], check=False)
    if not status.stdout.strip():
        print("No changes to commit (parquets already at origin).")
        return

    # Pre-commit invariant: every manifest source must be staged.
    staged = run(["git", "diff", "--cached", "--name-only"], check=False)
    staged_set = set(staged.stdout.strip().splitlines())
    for name in PARQUET_NAMES:
        rel = f"cell_sim/features/cache/{name}.parquet"
        if rel not in staged_set:
            print(f"[warn] {rel} NOT in staged set; check gitignore.")

    run(["git", "commit", "-m", COMMIT_MESSAGE])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    run(["git", "push", remote, BRANCH])
    print("Push complete.")


if OUTPUT_MODE == "download":
    _do_download()
elif OUTPUT_MODE == "drive":
    _do_drive()
elif OUTPUT_MODE == "github_pat":
    _do_github_pat()
else:
    print(f"unknown OUTPUT_MODE={OUTPUT_MODE!r}; "
          "choose one of download | drive | github_pat")


In [ ]:
# Cell 10 — final summary. Paste the output back to the review thread
# so the reviewer can verify the SHAs before running the downstream
# detector stack.
#
# Wall-time reporting is defensive: each extractor cell sets its own
# `*_wall` variable, but a kernel restart or partial run can leave
# some unset. We use globals().get(...) so the summary still prints
# the parquets even when one wall-time variable is missing.
from textwrap import dedent

manifest = CachedFeatureManifest.load(MANIFEST_PATH)
lines = ["## Tier-1 cache population summary", ""]
for name, entry in sorted(manifest.sources.items()):
    parquet = CACHE_DIR / f"{name}.parquet"
    if not parquet.exists():
        lines.append(f"### {name}  (v{entry['version']})")
        lines.append(f"- parquet: `{parquet}` [MISSING ON DISK]")
        lines.append(f"- sha256 (manifest): `{entry['sha256']}`")
        lines.append("")
        continue
    df = pd.read_parquet(parquet)
    lines.append(f"### {name}  (v{entry['version']})")
    lines.append(f"- parquet: `{parquet}`")
    lines.append(f"- rows: {entry['rows']}")
    lines.append(f"- sha256: `{entry['sha256']}`")
    lines.append(f"- created_at: {entry['created_at']}")
    lines.append(f"- first 3 locus_tags: {list(df.index[:3])}")
    sample_cols = list(df.columns[:5])
    lines.append(f"- first 5 columns: {sample_cols}")
    if sample_cols:
        first_values = df.iloc[0][sample_cols].to_dict()
        lines.append(f"- first-row values: {first_values}")
    lines.append("")

wall_sources = [
    ("ESM-2",    "esm_wall"),
    ("ESMFold",  "ef_wall"),
    ("MACE-OFF", "mace_wall"),
]
lines.append("### Wall times")
for label, var in wall_sources:
    val = globals().get(var)
    if isinstance(val, (int, float)):
        lines.append(f"- {label:9s} wall: {val:.1f}s")
    else:
        lines.append(f"- {label:9s} wall: (unset — cell did not complete)")

print("\n".join(lines))
